# RAG-Chatbot für PDF-Dokumente

Beantwortet Fragen zu einem PDF-Dokument mit LangChain, FAISS und einem Sprachmodell von OpenAI.

In [ ]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import ConversationalRetrievalChain

## Konfiguration

In [ ]:
PDF_PATH = "data/dokument.pdf"
MODEL_NAME = "gpt-3.5-turbo"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

# Eigener Index pro PDF, damit ein neues Dokument nicht den alten Index verwendet
INDEX_PATH = os.path.join("faiss_index", os.path.splitext(os.path.basename(PDF_PATH))[0])

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY fehlt in der .env-Datei.")

## Dokument laden und aufteilen

In [ ]:
def load_pdf(pdf_path):
    return PyPDFLoader(pdf_path).load()


# Die Überlappung verhindert, dass Sätze an den Chunk-Grenzen ihren Zusammenhang verlieren
def split_documents(documents, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_documents(documents)


documents = load_pdf(PDF_PATH)
chunks = split_documents(documents, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"{len(documents)} Seiten geladen, {len(chunks)} Chunks erstellt.")

## Vektordatenbank

In [ ]:
# Der Index wird gespeichert, weil das Erzeugen der Embeddings Zeit und API-Kosten verursacht
def build_or_load_vectorstore(chunks, embeddings, index_path):
    if os.path.exists(index_path):
        # Unbedenklich, da der Index lokal selbst erzeugt wurde
        return FAISS.load_local(index_path, embeddings, allow_dangerous_deserialization=True)
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(index_path)
    return vectorstore


embeddings = OpenAIEmbeddings()
vectorstore = build_or_load_vectorstore(chunks, embeddings, INDEX_PATH)

## Retrieval-Chain

In [ ]:
# temperature=0 hält die Antworten nah am Inhalt des Dokuments
# Der Chatverlauf ermöglicht Folgefragen, die sich auf frühere Antworten beziehen
def build_chain(vectorstore, model_name, temperature=0):
    llm = ChatOpenAI(model_name=model_name, temperature=temperature)
    return ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vectorstore.as_retriever(),
        return_source_documents=True,
    )


qa_chain = build_chain(vectorstore, MODEL_NAME)

## Chat

Mit `exit` oder `quit` beenden.

In [ ]:
def ask(question, chat_history):
    result = qa_chain.invoke({"question": question, "chat_history": chat_history})
    return result["answer"], result["source_documents"]


chat_history = []

while True:
    question = input("Frage: ")
    if question.lower() in ["exit", "quit"]:
        break

    answer, sources = ask(question, chat_history)
    print(f"Antwort: {answer}\n")
    print("Quellen:")
    for doc in sources:
        page = doc.metadata.get("page", 0) + 1
        print(f"- Seite {page}: {doc.page_content[:200]}...")

    chat_history.append((question, answer))